# Per-Second × Per-Cent Grid Search

Sweeps **every second** (50s → 420s observation window) and **every cent** (55c → 90c max entry price), **per market**.

**Tiered entry:** safe zone (last 50s) first; fallback to wider window only when no safe-zone tick qualifies.

- `max_entry_c` = the most the bot will pay for the favorite (ceiling)
- `obs_window_sec` = how far back the bot starts watching
- Floor is fixed at 51c (any favorite)

Uses numpy arrays instead of dicts for fast aggregation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import time

TICK_CSV       = "/content/drive/MyDrive/commod15min_ticks.csv"
OUT_OVERALL    = "/content/drive/MyDrive/sp_grid_overall.csv"
OUT_BY_ASSET   = "/content/drive/MyDrive/sp_grid_by_asset.csv"
OUT_BEST_ASSET = "/content/drive/MyDrive/sp_grid_best_per_asset.csv"

LOTS              = 10
SETTLE_WINDOW_SEC = 30.0
SAFE_SEC          = 50.0
SAFE_MINUTE       = 15.0 - SAFE_SEC / 60.0   # 14.1667
ENTRY_FLOOR       = 51.0

max_entry_grid = np.arange(55.0, 91.0, 1.0)   # 55c -> 90c  (36 values)
obs_grid_sec   = np.arange(50.0, 421.0, 1.0)   # 50s -> 420s (371 values)

print(f"max entry prices : {len(max_entry_grid)}  ({max_entry_grid[0]:.0f}c -> {max_entry_grid[-1]:.0f}c)")
print(f"obs windows      : {len(obs_grid_sec)}  ({obs_grid_sec[0]:.0f}s -> {obs_grid_sec[-1]:.0f}s)")
print(f"safe zone        : last {SAFE_SEC:.0f}s (fixed)")
print(f"entry floor      : {ENTRY_FLOOR:.0f}c (fixed)")
print(f"grid cells       : {len(max_entry_grid) * len(obs_grid_sec):,}")

In [ ]:
def determine_settlement(minute, ymid):
    sec_to_close = (15.0 - minute) * 60.0
    mask = sec_to_close <= SETTLE_WINDOW_SEC
    if mask.any():
        return ymid[mask].mean() > 50.0
    return ymid[-1] > 50.0


t0 = time.time()
print(f"loading {TICK_CSV} ...")
df = pd.read_csv(TICK_CSV, usecols=[
    "ticker", "asset", "yes_bid", "yes_ask", "yes_mid", "sec_to_close",
])
df = df.dropna(subset=["yes_mid", "sec_to_close"])
df["minute_in_cycle"] = 15.0 - df["sec_to_close"] / 60.0
df = df[(df["minute_in_cycle"] >= 0) & (df["minute_in_cycle"] <= 15)]

windows = []
for ticker, g in df.groupby("ticker", sort=False):
    g = g.sort_values("minute_in_cycle")
    minute = g["minute_in_cycle"].to_numpy()
    ymid   = g["yes_mid"].to_numpy(dtype=float)
    asset  = g["asset"].iloc[0]
    yes_won = determine_settlement(minute, ymid)
    windows.append((asset, minute, ymid, yes_won))

n_windows = len(windows)
print(f"loaded {n_windows} windows in {time.time()-t0:.1f}s")

In [ ]:
asset_list   = sorted(set(a for a, _, _, _ in windows))
asset_to_idx = {a: i for i, a in enumerate(asset_list)}
n_assets = len(asset_list)
n_me     = len(max_entry_grid)
n_obs    = len(obs_grid_sec)

# safe zone totals (same for every obs_window)
safe_ent = np.zeros((n_me, n_assets), dtype=np.int32)
safe_win = np.zeros((n_me, n_assets), dtype=np.int32)
safe_prf = np.zeros((n_me, n_assets), dtype=np.float64)

# fallback totals (vary by obs_window)
fb_ent = np.zeros((n_me, n_obs, n_assets), dtype=np.int32)
fb_win = np.zeros((n_me, n_obs, n_assets), dtype=np.int32)
fb_prf = np.zeros((n_me, n_obs, n_assets), dtype=np.float64)

report_every = max(1, n_windows // 20)

for wi, (asset, minute, ymid, yes_won) in enumerate(windows):
    ai = asset_to_idx[asset]
    n = len(minute)
    if n == 0:
        continue

    fav_price  = np.maximum(ymid, 100.0 - ymid)
    fav_is_yes = ymid >= 50.0

    # ── safe zone (last 50s) ──────────────────────────────
    s_mask = minute >= SAFE_MINUTE
    s_idx  = np.where(s_mask)[0]
    s_fav  = fav_price[s_idx]  if len(s_idx) else np.array([])
    s_yes  = fav_is_yes[s_idx] if len(s_idx) else np.array([], dtype=bool)

    safe_hit_mi = set()
    for mi, me in enumerate(max_entry_grid):
        if len(s_fav) == 0:
            break
        hits = (s_fav >= ENTRY_FLOOR) & (s_fav <= me)
        if not hits.any():
            continue
        fi  = int(np.argmax(hits))
        px  = float(s_fav[fi])
        iy  = bool(s_yes[fi])
        won = (iy and yes_won) or (not iy and not yes_won)
        pr  = (100.0 - px) * LOTS if won else 0.0

        safe_ent[mi, ai] += 1
        if won:
            safe_win[mi, ai] += 1
            safe_prf[mi, ai] += pr
        safe_hit_mi.add(mi)

    # ── fallback (ticks before safe zone) ─────────────────
    missing = [mi for mi in range(n_me) if mi not in safe_hit_mi]
    if not missing:
        if (wi + 1) % report_every == 0:
            print(f"  {wi+1}/{n_windows} ({100*(wi+1)/n_windows:.0f}%) -- {time.time()-t0:.0f}s")
        continue

    ps_mask = minute < SAFE_MINUTE
    ps_idx  = np.where(ps_mask)[0]
    if len(ps_idx) == 0:
        if (wi + 1) % report_every == 0:
            print(f"  {wi+1}/{n_windows} ({100*(wi+1)/n_windows:.0f}%) -- {time.time()-t0:.0f}s")
        continue

    ps_minute = minute[ps_idx]
    ps_fav    = fav_price[ps_idx]
    ps_yes    = fav_is_yes[ps_idx]

    for mi in missing:
        me = max_entry_grid[mi]
        qualifying = (ps_fav >= ENTRY_FLOOR) & (ps_fav <= me)
        q_idx = np.where(qualifying)[0]
        if len(q_idx) == 0:
            continue

        q_min = ps_minute[q_idx]     # sorted ascending
        q_fav = ps_fav[q_idx]
        q_yes = ps_yes[q_idx]
        n_q   = len(q_min)

        # sec before close for each qualifying tick
        tick_sbc = (15.0 - q_min) * 60.0   # descending

        # assign obs_sec ranges to each qualifying tick
        for qi in range(n_q - 1, -1, -1):
            lower = tick_sbc[qi]
            upper = tick_sbc[qi - 1] if qi > 0 else obs_grid_sec[-1] + 1

            lo_i = int(np.searchsorted(obs_grid_sec, lower, side='left'))
            hi_i = int(np.searchsorted(obs_grid_sec, upper, side='left'))
            lo_i = max(lo_i, 1)          # skip index 0 (obs=50, no fallback)
            if lo_i >= hi_i:
                continue

            px  = float(q_fav[qi])
            iy  = bool(q_yes[qi])
            won = (iy and yes_won) or (not iy and not yes_won)
            pr  = (100.0 - px) * LOTS if won else 0.0

            fb_ent[mi, lo_i:hi_i, ai] += 1
            if won:
                fb_win[mi, lo_i:hi_i, ai] += 1
                fb_prf[mi, lo_i:hi_i, ai] += pr

    if (wi + 1) % report_every == 0:
        print(f"  {wi+1}/{n_windows} ({100*(wi+1)/n_windows:.0f}%) -- {time.time()-t0:.0f}s")

print(f"sweep done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── combine safe + fallback into totals ───────────────────
total_ent = safe_ent[:, np.newaxis, :] + fb_ent    # (n_me, n_obs, n_assets)
total_win = safe_win[:, np.newaxis, :] + fb_win
total_prf = safe_prf[:, np.newaxis, :] + fb_prf

# ── per-asset DataFrame ───────────────────────────────────
rows = []
for mi in range(n_me):
    me = max_entry_grid[mi]
    for oi in range(n_obs):
        obs = obs_grid_sec[oi]
        for ai in range(n_assets):
            ent = int(total_ent[mi, oi, ai])
            if ent == 0:
                continue
            rows.append({
                'obs_window_sec': int(obs),
                'max_entry_c':    me,
                'asset':          asset_list[ai],
                'entries':        ent,
                'wins':           int(total_win[mi, oi, ai]),
                'total_profit_c': round(float(total_prf[mi, oi, ai]), 1),
                'safe_entries':   int(safe_ent[mi, ai]),
                'safe_wins':      int(safe_win[mi, ai]),
                'safe_profit_c':  round(float(safe_prf[mi, ai]), 1),
                'fb_entries':     int(fb_ent[mi, oi, ai]),
                'fb_wins':        int(fb_win[mi, oi, ai]),
                'fb_profit_c':    round(float(fb_prf[mi, oi, ai]), 1),
            })
detail = pd.DataFrame(rows)
detail['win_rate'] = (detail['wins'] / detail['entries']).round(4)
detail.to_csv(OUT_BY_ASSET, index=False)
print(f"wrote {OUT_BY_ASSET} ({len(detail):,} rows)")

# ── overall (all assets summed) ──────────────────────────
tot_ent_all = total_ent.sum(axis=2)   # (n_me, n_obs)
tot_win_all = total_win.sum(axis=2)
tot_prf_all = total_prf.sum(axis=2)
saf_ent_all = safe_ent.sum(axis=1)    # (n_me,)
saf_win_all = safe_win.sum(axis=1)
saf_prf_all = safe_prf.sum(axis=1)
fb_ent_all  = fb_ent.sum(axis=2)      # (n_me, n_obs)
fb_win_all  = fb_win.sum(axis=2)
fb_prf_all  = fb_prf.sum(axis=2)

orows = []
for mi in range(n_me):
    me = max_entry_grid[mi]
    for oi in range(n_obs):
        ent = int(tot_ent_all[mi, oi])
        if ent == 0:
            continue
        orows.append({
            'obs_window_sec': int(obs_grid_sec[oi]),
            'max_entry_c':    me,
            'entries':        ent,
            'wins':           int(tot_win_all[mi, oi]),
            'total_profit_c': round(float(tot_prf_all[mi, oi]), 1),
            'safe_entries':   int(saf_ent_all[mi]),
            'safe_wins':      int(saf_win_all[mi]),
            'safe_profit_c':  round(float(saf_prf_all[mi]), 1),
            'fb_entries':     int(fb_ent_all[mi, oi]),
            'fb_wins':        int(fb_win_all[mi, oi]),
            'fb_profit_c':    round(float(fb_prf_all[mi, oi]), 1),
        })
overall = pd.DataFrame(orows)
overall['win_rate'] = (overall['wins'] / overall['entries']).round(4)
overall['fb_win_rate'] = np.where(
    overall['fb_entries'] > 0,
    (overall['fb_wins'] / overall['fb_entries']).round(4), 0.0
)
overall = overall.sort_values('total_profit_c', ascending=False)
overall.to_csv(OUT_OVERALL, index=False)
print(f"wrote {OUT_OVERALL} ({len(overall):,} rows)")

# ── best combo per asset ─────────────────────────────────
best_asset = (
    detail
    .sort_values('total_profit_c', ascending=False)
    .groupby('asset', sort=False)
    .first()
    .reset_index()
    .sort_values('asset')
)
best_asset.to_csv(OUT_BEST_ASSET, index=False)
print(f"wrote {OUT_BEST_ASSET} ({len(best_asset)} rows)")

print(f"\ntotal time: {(time.time()-t0)/60:.1f} min")

In [ ]:
cols = [
    'obs_window_sec', 'max_entry_c',
    'entries', 'wins', 'win_rate', 'total_profit_c',
    'safe_entries', 'safe_wins', 'safe_profit_c',
    'fb_entries', 'fb_wins', 'fb_profit_c',
]

print("TOP 20 COMBOS BY TOTAL PROFIT (all assets combined)")
print("="*100)
overall[cols].head(20)

In [ ]:
print("BEST COMBO PER ASSET")
print("="*100)
best_asset[['asset'] + cols]

In [ ]:
print("PROGRESSION: best max_entry at each obs_window (overall)")
print("Shows how total profit changes as the observation window widens")
print("="*100)

best_per_obs = (
    overall
    .sort_values('total_profit_c', ascending=False)
    .groupby('obs_window_sec', sort=False)
    .first()
    .reset_index()
    .sort_values('obs_window_sec')
)

# show every 10 seconds for readability
show = best_per_obs[best_per_obs['obs_window_sec'] % 10 == 0]
show[cols]